<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/Project4Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Travel Assistant with a ReAct Agent and Supervisor
1. Problem Statement

A travel assistant must answer factual questions by calling tools instead of guessing. This project builds:

Two tools, get_weather(city) and find_flight(source, destination).
Two ReAct agents built with create_react_agent(), one per tool. The agent decides on its own when to call its tool.
A supervisor built with create_supervisor() that routes each question to the right agent and checks the answer against the tool output before replying.

The tools return fixed mock data so the demo is repeatable. In a real project you would swap in a weather or flight API.

User → supervisor ──weather question──► weather_agent → get_weather ─┐
                  └─flight question───► flight_agent  → find_flight ─┤
                                                                     ▼
                                    supervisor verifies → final answer

Packages to Install

langgraph	create_react_agent and the graph runtime
langgraph-supervisor	create_supervisor
langchain-core	the @tool decorator, message types
langchain-openai	ChatOpenAI

In [2]:
!pip install -q langgraph langgraph langgraph-supervisor langchain-core langchain-openai

In [10]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')


In [14]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent



In [12]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [5]:
@tool
def get_weather(city:str) -> str:
  """Get the current weather in a given city"""
  print(f"Tool invoked: get_weather for city {city}")
  forecasts = {
        "delhi": "31°C, partly cloudy",
        "mumbai": "29°C, humid with light rain",
        "bengaluru": "26°C, clear skies",
    }
  return forecasts.get(city.strip().lower(), "no forecast available")

In [6]:
@tool
def find_flight(source:str, destination:str) -> str:
  """Find a flight from source to destination"""
  print(f"Tool invoked: find_flight from {source} to {destination}")
  flights = {
      ("delhi", "mumbai"): "AI-101, departs 08:30, arrives 10:40, ₹6,200",
      ("bengaluru", "delhi"): "6E-204, departs 07:15, arrives 10:00, ₹7,100",
  }
  return flights.get((source.strip().lower(), destination.strip().lower()), "no flights found")



In [19]:
weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    name="weather_agent",
    system_prompt=(
        "You answer weather questions. Always use the get_weather tool. "
        "Report only what the tool returns."
    )
)


In [21]:
flight_agent = create_agent(
    model=llm,
    tools=[find_flight],
    name="flight_agent",
    system_prompt="You answer flight questions. Always use the find_flight tool. "
        "Report only what the tool returns.",
)

In [22]:
from langgraph_supervisor import create_supervisor
supervisor = create_supervisor(
    agents = [weather_agent, flight_agent],
    model = llm,
    output_mode="full_history",
    prompt=(
        "You are a supervisor for a travel assistant. Route weather questions to "
        "weather_agent and flight questions to flight_agent. "
        "When a worker replies, VERIFY that its answer is based on tool output and "
        "does not contain invented details. If it checks out, reply to the user with "
        "the answer on the first line, then on a new line write exactly: "
        "Supervisor: Output verified."
    ),
).compile()

In [27]:
def ask(question):
  print(f"\nUser: {question}")
  result = supervisor.invoke({"messages":[("user", question)]})
  print(f"Agent: {result['messages'][-1].content}")


ask("What's the weather in mangaluru tomorrow and also can you add in a date as well?")


User: What's the weather in mangaluru tomorrow and also can you add in a date as well?
Tool invoked: get_weather for city Mangaluru
Agent: I couldn't find a weather forecast for Mangaluru tomorrow.

Supervisor: Output verified.
